# Model Evaluation and Selection

This notebook refreshes model evaluation and selection around the following principle:

> **Any use of data that influences a modeling decision is part of the learning process. Data held out for final reporting must not participate in that learning process.**


By the end of the notebook, you should be able to:

- Separate data used for learning from data used for final reporting;
- Recognize hyperparameter and model selection as learning;
- Evaluate complete pipelines without leaking information; and
- Distinguish cross-validation for selection from nested cross-validation for evaluation.

We use the following terminology throughout:

- **Training set:** the portion available for learning.
- **Reporting set:** the untouched portion used to report final performance. This is conventionally called the *test set*. We emphasize its role by calling it the reporting set.
- **Fitting set:** data used to estimate model parameters inside a training split.
- **Tuning set:** data used to choose hyperparameters or candidate models. This is conventionally often called the *validation set*.

***Disclaimer:*** This terminology is adapted from Prof. Pradeep Raamana, University of Pittsburgh (see https://crossinvalidation.com/2020/06/04/unambiguous-terminology-for-data-splits-in-nested-cross-validation-cv-training-tuning-and-reporting-sets/). The images included in this notebook were adapted using an LLM from those provided at the linked source.

## 1. The training/reporting contract

In the previous class, we introduced the training/holdout contract. We will retain this contract but change the terminology by calling the **holdout set** the **reporting set**. Only the terminology changes; the underlying concept remains the same.

The **reporting set** is an untouched, held-out portion of the data used to report final performance. This is conventionally called the *test set*. We emphasize its role by calling it the reporting set.

```text
Full dataset
      |
      +-- Training set  -> learning happens here
      |
      +-- Reporting set -> final performance estimate
```

> **The contract:** The reporting set must not influence any decision about preprocessing, feature selection, model family, hyperparameters, thresholds, or any other component of the learning procedure.

<img src="cv1.png" width="800">

This dataset contains a cohort of adult patients assessed for cardiovascular risk. It includes demographic characteristics, lifestyle factors, vital signs, medical history, and laboratory measurements collected at baseline. Two outcomes are available: systolic_bp_12m, a continuous measure of systolic blood pressure at 12-month follow-up, and cvd_event_5yr, a binary indicator of whether a cardiovascular event occurred within five years.

In [ ]:
import os
import warnings

warnings.filterwarnings(
    "ignore",
    message=r"Pandas requires version .* of 'bottleneck'",
    category=UserWarning,
)
os.environ["PYTHONWARNINGS"] = "ignore:Pandas requires version"

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.base import clone
from sklearn.datasets import make_regression
from sklearn.feature_selection import SelectKBest, f_regression, f_classif
from sklearn.impute import SimpleImputer
from sklearn.linear_model import Lasso, Ridge
from sklearn.metrics import mean_squared_error
from sklearn.model_selection import (
    GridSearchCV, KFold, cross_val_score, train_test_split
)
from sklearn.pipeline import Pipeline

df = pd.read_csv('./cardiovascular_dataset.csv')

X = df.drop(columns=["cvd_event_5yr", "systolic_bp_12m", "diabetes_status"])
Y = df[["cvd_event_5yr", "systolic_bp_12m"]]

# HERE COMES CODE: Create training and reporting sets using train_test_split, with a reporting-set size of 0.2 and random_state = 0
X_train, X_report, Y_train, Y_report = None, None, None, None
print(f"Training observations: {len(X_train)}")
print(f"Reporting observations: {len(X_report)}")

From now until the final report, `X_report` and `y_report` are off limits.

> **Key idea:** A dataset's role is determined by how it is used. A reporting set becomes part of learning the moment it influences a choice.

### Putting everything together

We are going to work with two different predictive pipelines: one for regression and another for classification. In both cases, we will use k-nearest neighbours.

The pipelines will also include imputation of missing values, scaling of numeric features, encoding of categorical data, and feature selection.



In [ ]:
numeric_idxs = np.where((X.dtypes == "int64") | (X.dtypes == "float64"))[0]
categorical_idxs = [1,2,3,5]
ordinal_idxs = [4]

In [ ]:
import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler, OrdinalEncoder

# HERE COMES CODE: Create a transformer that imputes the numeric features with the median and then standardizes them,
#                  plus a one-hot encoder for the nominal data and an ordinal encoder for the ordinal data.

preprocessor = None

In [ ]:
from sklearn.neighbors import KNeighborsClassifier, KNeighborsRegressor
my_regressor  = Pipeline([
    ("preprocessing", preprocessor),
    ("selector", SelectKBest(k=10, score_func=f_regression)),
    ("knn_regressor", KNeighborsRegressor()),
])

# HERE COMES CODE: Create an equivalent pipeline ending with a kNN classifier
my_classifier  = None

### Reporting performance

> Always include more than one performance metric. 

For **regression** tasks:

- **Coefficient of determination ($R^2$)**: This is the preferred metric for assessing predictive performance, alongside performance metric that carry units for better interpretation (see next). It measures the proportion of variability in the outcome that is explained by the model. Higher values are better, with $R^2 = 1$ indicating perfect predictions.
$$
R^2 = 1 - \frac{\sum_{i=1}^{n}(y_i-\hat{y}_i)^2}
{\sum_{i=1}^{n}(y_i-\bar{y})^2}
$$
- **Mean Squared Error (MSE)**: Measures the average squared difference between the observed and predicted values. Larger errors are penalized more heavily because they are squared. Lower values are better.
$$
MSE = \frac{1}{n}\sum_{i=1}^{n}(y_i-\hat{y}_i)^2
$$

- **Mean Absolute Error (MAE)**: Measures the average absolute difference between the observed and predicted values. It is expressed in the same units as the outcome variable. Lower values are better.
$$
MAE = \frac{1}{n}\sum_{i=1}^{n}|y_i-\hat{y}_i|
$$

- **Pearson Correlation ($r$)**: Measures the strength of the linear association between the observed and predicted values. It ranges from $-1$ to $1$, with values closer to $1$ indicating a stronger positive association. However, correlation alone does not measure prediction accuracy.
$$
r =
\frac{
\sum_{i=1}^{n}(y_i-\bar{y})(\hat{y}_i-\overline{\hat{y}})
}{
\sqrt{
\sum_{i=1}^{n}(y_i-\bar{y})^2
\sum_{i=1}^{n}(\hat{y}_i-\overline{\hat{y}})^2
}
}
$$

where $y_i$ is the observed value, $\hat{y}_i$ is the predicted value, $\bar{y}$ is the mean of the observed values, and $n$ is the number of observations.

In [ ]:
# HERE COMES CODE: Import metrics to use: r2_score, mean_absolute_error
# HERE COMES CODE: Fit the regression pipeline using the training data and continuous outcome, Y_train.loc[:, "systolic_bp_12m"]

y_pred = None # HERE COMES CODE: Predict using the reporting set

# HERE COMES CODE: Calculate the different metrics
r2 = None
mae = None
corr = None

print(f"Reporting performance: R2 = {r2:.3f}, MAE = {mae:.3f} mmHg, Pearson's r = {corr:.3f}")


For classification:

- **Accuracy**: Measures the proportion of observations that are classified correctly. Higher values are better, although accuracy can be misleading when the classes are highly imbalanced.

- **Confusion Matrix**: Summarizes the number of correct and incorrect predictions for each class.


|                 | Predicted Positive | Predicted Negative |
|-----------------|-------------------:|-------------------:|
| Actual Positive | TP                 | FN                 |
| Actual Negative | FP                 | TN                 |

where:

TP = true positives  
TN = true negatives  
FP = false positives  
FN = false negatives

- **ROC – Area Under the Curve (ROC-AUC)**: The ROC curve plots the True Positive Rate (Sensitivity or Recall) against the False Positive Rate across different classification thresholds. The area under the curve (AUC) summarizes the overall ability of the model to distinguish between the two classes. A value of $0.5$ corresponds to random classification, while $1.0$ represents perfect discrimination. Higher values are better.
  
$$ 
\begin{align}
\text{TPR} & = \frac{\text{TP}}{\text{TP} + \text{FN}}\\
\\
\text{FPR} & = \frac{\text{FP}}{\text{FP} + \text{TN}}
\end{align}
$$


- **F1-score**: Combines precision and recall into a single measure. It is particularly useful when the positive class is relatively uncommon or when both false positives and false negatives are important. Higher values are better.
$$
\text{F1} =
\frac{2 \times \text{TP}}
{2 \times \text{TP} + \text{FP} + \text{FN}}
$$

Or, equivalently:

$$
\text{F1} =
2 \times
\frac{\text{Precision} \times \text{Recall}}
{\text{Precision} + \text{Recall}}
$$

with:

$$
\text{Precision} =
\frac{\text{TP}}
{\text{TP} + \text{FP}}
$$

$$
\text{Recall} =
\frac{\text{TP}}
{\text{TP} + \text{FN}}
$$

In [ ]:
from sklearn.metrics import accuracy_score, confusion_matrix

my_classifier.fit(X_train, Y_train.loc[:, "cvd_event_5yr"])
y_pred = my_classifier.predict(X_report)

acc = accuracy_score(Y_report.loc[:, "cvd_event_5yr"], y_pred)
print(f"Reporting performance: Accuracy = {acc:.3f}")

Accuracy could be misleading because our outcome is heavily imbalanced:

In [ ]:
Y.loc[:, "cvd_event_5yr"].value_counts()/Y.loc[:, "cvd_event_5yr"].shape[0]*100

In [ ]:
from sklearn.metrics import ConfusionMatrixDisplay
conf_mat = confusion_matrix(Y_report.loc[:, "cvd_event_5yr"], y_pred)
ConfusionMatrixDisplay(conf_mat).plot()

In these cases, the area under the receiver operating characteristic curve (ROC AUC) and the F1-score are more representative and informative metrics.

In [ ]:
# HERE COMES CODE: Import RocCurveDisplay to display the ROC curve and its AUC

y_prob = None # HERE COMES CODE: Calculate the predicted class probabilities
display = RocCurveDisplay.from_predictions(
    Y_report.loc[:, "cvd_event_5yr"],
    y_prob[:, -1],
    plot_chance_level=True,
    despine=True,
)

In [ ]:
from sklearn.metrics import f1_score

f1_score(Y_report.loc[:, "cvd_event_5yr"], y_pred)

**Exercise:** Calculate the F1-score using the confusion matrix values.

In [ ]:
# YOUR CODE HERE

## 2. One split is only one split

This result comes from only one training/reporting split. What happens if we use a different split?

In [ ]:
X_train, X_report, Y_train, Y_report = train_test_split(
    X, Y, test_size=0.20, random_state=10
)

my_regressor.fit(X_train, Y_train.iloc[:,1])
y_pred = my_regressor.predict(X_report)

print("Reporting performance for this split: R2 = "
      f"{r2_score(Y_report.iloc[:,1], y_pred):.3f}, mae = {mean_absolute_error(Y_report.iloc[:,1], y_pred):.3f} mmHg, "
      f"Pearson's r = {np.corrcoef(Y_report.iloc[:,1], y_pred)[0,1]:.3f}")


As you can see, the result has changed. This happens because our finite dataset has inherent variability, so the result depends on how we split the data.

Let's repeat this procedure with more splits to see the variation more clearly:

In [ ]:
from sklearn.metrics import roc_auc_score
n_splits = 10
split_results = []
for seed in range(n_splits):
    X_train, X_report, Y_train, Y_report = train_test_split(
        X, Y, test_size=0.20, random_state=seed
    )

    model_reg = clone(my_regressor).fit(X_train, Y_train.iloc[:,1])
    r2 = r2_score(Y_report.iloc[:,1], model_reg.predict(X_report)) 
    
    model_clf = clone(my_classifier).fit(X_train, Y_train.iloc[:,0])
    auc = roc_auc_score(Y_report.iloc[:,0], model_clf.predict_proba(X_report)[:,-1]) 
    split_results.append({"random_seed": seed, "reporting_R2": r2, "reporting_AUC": auc})


split_results = pd.DataFrame(split_results)

fig, axs = plt.subplots(ncols=2, figsize=(10,7))

split_results["reporting_R2"].plot.bar(
    xlabel="Random seed", ylabel="Reporting R2", legend=False, ax=axs[0]
)
axs[0].set_title("Regression")
split_results["reporting_AUC"].plot.bar(
    xlabel="Random seed", ylabel="reporting ROC AUC", legend=False, ax=axs[1]
)
axs[1].set_title("Classification")
plt.suptitle("The estimated performance changes with the split")
plt.tight_layout()
plt.show()

The complete pipelines have not changed, but the estimated performance changes because different observations are assigned to the training and reporting sets.

```text
One train/reporting split
            |
            v
Performance depends on the split
            |
            v
Resampling
```

> **Key idea:** A single split produces a single, split-dependent estimate. Resampling lets us examine performance across several plausible partitions.

## 3. Cross-validation

**Cross-validation** is a technique in which we resample without replacement so that each data point is assigned to a specific subset, or fold, once in each partition.

The **bootstrap** repeatedly samples observations with replacement and examines the resulting variation. Both address sampling/splitting variability.

We will focus on cross-validation because it is one of the most widely used techniques for reporting model performance and because it guarantees that each point is used for reporting.

Typical cross-validation procedures include:

- **K-Fold Cross-Validation**. The dataset is divided into $K$ approximately equal-sized folds. The model is trained on $K-1$ folds and evaluated on the remaining fold. This process is repeated $K$ times so that each fold is used once as the holdout set. The final performance is usually reported as the average across all folds.
$$
\text{CV Score} =
\frac{1}{K}\sum_{k=1}^{K} M_k
$$

where $M_k$ is the evaluation metric obtained on fold $k$. A common choice is $K=5$ or $K=10$.

- **Stratified Cross-Validation**. Similar to K-Fold Cross-Validation, but each fold preserves approximately the same class proportions as the complete dataset. This is especially useful for classification problems, particularly when the classes are imbalanced.

- **Leave-One-Out Cross-Validation (LOOCV)**: A special case of K-Fold Cross-Validation where the number of folds is equal to the number of observations. Therefore, for each iteration, the model is trained using $n-1$ observations and evaluated on the single observation that was held out. The process is repeated until every observation has been used once as the holdout observation.

<img src="cv2.png" width="800">

In [ ]:
from sklearn.model_selection import cross_val_score, KFold, StratifiedKFold, LeaveOneOut

cv10 = KFold(n_splits=10, random_state=1234, shuffle=True)

cv_results  = []
for fold_id, (train_index, report_index) in enumerate(cv10.split(X, Y)):
    X_train, X_report = X.iloc[train_index,:], X.iloc[report_index,:]
    Y_train, Y_report = Y.iloc[train_index], Y.iloc[report_index]

    model_reg = clone(my_regressor).fit(X_train, Y_train.iloc[:,1])
    r2 = r2_score(Y_report.iloc[:,1], model_reg.predict(X_report)) 
    
    model_clf = clone(my_classifier).fit(X_train, Y_train.iloc[:,0])
    auc = roc_auc_score(Y_report.iloc[:,0], model_clf.predict_proba(X_report)[:,-1]) 
    cv_results.append({"fold ID": fold_id, "reporting_R2": r2, "reporting_AUC": auc})

cv_results = pd.DataFrame(cv_results)

print(cv_results)
print(cv_results[["reporting_R2", "reporting_AUC"]].describe())

In [ ]:
from sklearn.model_selection import cross_validate
from sklearn.metrics import make_scorer

def pearson_score(y_true, y_pred):
    return np.corrcoef(y_true, y_pred)[0,1]

# Combine standard metrics and custom metrics in a dictionary
scoring_metrics = {
    'r2': 'r2',
    'mae': 'neg_mean_absolute_error',
    'pearson': make_scorer(pearson_score),
}

res_reg_cv = cross_validate(my_regressor, X, Y.iloc[:, 1], cv=cv10, scoring=scoring_metrics)

print("Reporting cross-validated performance: R2 = ",
      f"{res_reg_cv['test_r2'].mean():.3f}, mae = {-res_reg_cv['test_mae'].mean():.3f} mmHg, "
      f"Pearson's r = {res_reg_cv['test_pearson'].mean():.3f}")


**Exercise**: Do the same as above, i.e., cross-validation, but for our classification pipeline. Needless to say you need to use the appropriate performance metrics for a classification task.

In [ ]:
#YOUR CODE HERE

**Question:** How does the choice of $K$ in K-fold cross-validation affect the bias and variance of the performance estimate?

## 4. Hyperparameter selection is learning

During the fitting process, some **parameters** are estimated, such as regression coefficients and the means used for imputation.

**Hyperparameters** control how fitting happens. 

For example, in our pipeline, these include the imputation strategy (`mean` or `median`), the number of selected features (5, 10, or 20), and the number of neighbours $k = (1, 2, \dots)$.

> Hyperparameters can belong to any pipeline stage—not only the final estimator.

If we compare several $k$ values and choose the one with the smallest error, we have learned from the data that produced those errors. 

Therefore, that data cannot be the reporting set. We need a split *inside* the training set:

```text
Full dataset
|
+-- Training set
|     |
|     +-- Fitting set -> estimate model parameters
|     +-- Tuning set  -> select hyperparameters
|
+-- Reporting set     -> remains untouched
```

<img src="cv3.png" width="800">

In [ ]:
X_fit, X_tune, Y_fit, Y_tune = train_test_split(
    X_train, Y_train, test_size=0.25, random_state=1
)

k_results = []
for k in [1, 5, 10, 20]:
    candidate = clone(my_regressor).set_params(**{"knn_regressor__n_neighbors":k})

    candidate.fit(X_fit, Y_fit.iloc[:,1])
    tuning_r2 = r2_score(Y_tune.iloc[:,1], candidate.predict(X_tune))
    k_results.append({"k": k, "tuning_R2": tuning_r2})

k_results = pd.DataFrame(k_results).sort_values("tuning_R2")
print(k_results)


selected_k = k_results.iloc[k_results["tuning_R2"].argmax()]["k"]
print(f"Selected k: {selected_k:g}")

Notice what happened: fitting uses the nearest neighbors after tuning the selected $k$. Both are learning. We still have not inspected the reporting performance.

> **Key idea:** Hyperparameter selection is not preparation for learning—it is part of learning.

## 5. The tuning split has the same problem

Why should we trust one arbitrary fitting/tuning split? Let us repeat the manual selection using several splits. 

In [ ]:
from sklearn.base import clone
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score
import pandas as pd

results = []

for seed in [1, 2, 3, 4, 5, 10, 20, 50]:

    # Different fitting/tuning split for each seed
    X_fit, X_tune, Y_fit, Y_tune = train_test_split(X_train,Y_train, test_size=0.25, random_state=seed)

    for k in [1, 5, 10, 20]:

        candidate = clone(my_regressor).set_params(**{"knn_regressor__n_neighbors": k})
        candidate.fit(X_fit, Y_fit.iloc[:, 1])
        predictions = candidate.predict(X_tune)
        tuning_r2 = r2_score(Y_tune.iloc[:, 1], predictions)

        results.append({"split": seed,"k": k,"tuning_R2": tuning_r2})

results = pd.DataFrame(results)

selected_by_split = (
    results.loc[
        results.groupby("split")["tuning_R2"].idxmax()
    ]
    .sort_values("split")
    .reset_index(drop=True)
)

selected_by_split

As you can see, the selected $k$ may depend on the tuning split, just as reported performance depended on the first split.

**We can also cross-validate this tuning process!**

```text
One fitting/tuning split -> selected model depends on split -> cross-validated selection

Training data
Fold 1 -> tuning/evaluation
Fold 2 -> tuning/evaluation
Fold 3 -> tuning/evaluation
Fold 4 -> tuning/evaluation
Fold 5 -> tuning/evaluation
             |
             v
Average performance for each candidate -> select candidate
```

Each observation takes a turn in a tuning role. Candidate performance is averaged across folds, reducing dependence on one arbitrary partition.


<img src="cv4.png" width="800">

In `scikit-learn`, it is very easy to run this process by using `GridSearchCV`:

In [ ]:
inner_cv = KFold(n_splits=5, shuffle=True, random_state=1234)
candidate_grid = {"knn_regressor__n_neighbors":[1, 5, 10, 20]}

search = GridSearchCV(
    estimator=my_regressor,
    param_grid=candidate_grid,
    scoring="r2",
    cv=inner_cv,
    n_jobs=-1,
)
search.fit(X_train, Y_train.iloc[:, 1])

print(f"Best cross-validated R2: {search.best_score_:.2f}, corresponding to {search.best_params_}")

`GridSearchCV` automates the same comparison logic we wrote manually. As the full pipeline is its estimator, imputation and feature selection are refitted within every fold.

> **Key idea:** Cross-validated selection evaluates each complete candidate across multiple fitting/tuning partitions.

**Exercise:** Repeat the preceding process for our classification task.

In [ ]:
# YOUR RESPONSE HERE

## 6. Model selection applies to the complete learning process, not only the final estimator

We have tuned our pipeline to select the best $k$ in k-NN, but this is not the only hyperparameter we could have learned.

For example, we could have decided to tune the number of features to keep in the feature selection step:

In [ ]:
inner_cv = KFold(n_splits=5, shuffle=True, random_state=1234)
candidate_grid = {"selector__k":[5, 10, 15, 20]}

search = GridSearchCV(
    estimator=my_regressor,
    param_grid=candidate_grid,
    scoring="r2",
    cv=inner_cv,
    n_jobs=-1,
)
search.fit(X_train, Y_train.iloc[:, 1])

print(f"Best cross-validated R2: {search.best_score_:.2f}, corresponding to {search.best_params_}")

**Exercise:** Repeat the preceding process for our classification task.

In [ ]:
# YOUR RESPONSE HERE

From the perspective of selection, these are also competing candidate configurations.

A complete candidate is larger than its final estimator. For example, we could tune $k$ and the number of features to select:

In [ ]:
inner_cv = KFold(n_splits=5, shuffle=True, random_state=1234)
candidate_grid = {"selector__k":[5, 10, 15, 20], 
                  "knn_regressor__n_neighbors":[1, 5, 10, 20]}

search = GridSearchCV(
    estimator=my_regressor,
    param_grid=candidate_grid,
    scoring="r2",
    cv=inner_cv,
    n_jobs=-1,
)
search.fit(X_train, Y_train.iloc[:, 1])

print(f"Best cross-validated R2: {search.best_score_:.2f}, corresponding to {search.best_params_}")

## 7. Reporting after cross-validated model selection

We can now complete the workflow:

```text
Full dataset
|
+-- Training set
|      +-- Cross-validation
|             +-- compare complete candidate pipelines
|             +-- choose preprocessing, features, family, hyperparameters
|
+-- Reporting set
       +-- evaluate the final selected learning procedure once
```

This is a **reporting holdout plus cross-validation for tuning/model selection**. 

By default, `GridSearchCV` refits the chosen configuration on the full training set, so `search.best_estimator_` is ready for the one final evaluation.

In [ ]:
final_pipeline = search.best_estimator_
reporting_preds = final_pipeline.predict(X_report)
reporting_r2 = r2_score(Y_report.iloc[:,1], reporting_preds)

print(final_pipeline)
print(f"Final reporting R2: {reporting_r2:.2f}")

We report this result; we do not use it to revise the pipeline. Revising after seeing it would turn the reporting set into additional tuning data.

> **Key idea:** The final evaluation is final because its result does not trigger another modeling decision.

## 8. From cross-validation to nested cross-validation

Cross-validation used above answers: **Which candidate should we choose?** But what if we want resampling to estimate the performance of the *entire selection procedure*?

There are now two distinct problems:

- **Inner problem:** choose the model and hyperparameters.
- **Outer problem:** estimate how well the complete learning-and-selection procedure generalizes.

```text
Outer fold
|
+-- Outer training data
|      +-- Inner cross-validation
|             +-- compare candidate pipelines
|             +-- select model and hyperparameters
|
+-- Outer evaluation fold
       +-- evaluate the selected model

Outer fold 1 -> inner tuning -> outer score
Outer fold 2 -> inner tuning -> outer score
Outer fold 3 -> inner tuning -> outer score
... -> aggregate outer scores
```

The outer evaluation fold must not influence inner selection.

<img src="cv5.png" width="800">

In [ ]:
nested_inner_cv = KFold(n_splits=5, shuffle=True, random_state=1234)
outer_cv = KFold(n_splits=5, shuffle=True, random_state=0)

candidate_grid = {"knn_regressor__n_neighbors":[1, 5, 10, 20]}

inner_search = GridSearchCV(estimator=my_regressor, 
                            param_grid=candidate_grid,
                            scoring="r2",
                            cv=nested_inner_cv,
                            n_jobs=-1,
                           )


# Combine standard metrics and custom metrics in a dictionary
scoring_metrics = {
    'r2': 'r2',
    'mae': 'neg_mean_absolute_error',
    'pearson': make_scorer(pearson_score),
}

res_reg_cv = cross_validate(inner_search, X, Y.iloc[:, 1], cv=outer_cv, scoring=scoring_metrics, return_estimator=True)


print("Reporting cross-validated performance: R2 = ",
      f"{res_reg_cv['test_r2'].mean():.3f}, mae = {-res_reg_cv['test_mae'].mean():.3f} mmHg, "
      f"Pearson's r = {res_reg_cv['test_pearson'].mean():.3f}")

Since in each outer fold we are using different data for training/tuning, the selected $k$ may vary. This is fine; $k$ is a hyperparameter, chosen from the available training data, not a fixed parameter of the underlying population.

**What we are evaluating is the performance of the whole pipeline design, not one particular configuration.**

In [ ]:
for outer_fold_id, reg in enumerate(res_reg_cv["estimator"]):
    print("outer fold ID = ", outer_fold_id, "selected params", reg.best_params_)

Inner CV performs model selection. Outer CV estimates the performance of the full learning-and-selection procedure. Nested CV is therefore not merely “CV with more folds”: the two levels have different roles.

> **Key idea:** Whenever resampling is used both to select a candidate and to estimate the performance of that selection procedure, those roles require separate inner and outer loops.

## 9. Two valid evaluation designs

### Design A: Reporting holdout + cross-validation for tuning

```text
Training -> CV model selection
Reporting -> one final evaluation
```

This is useful when a dedicated reporting set is available. For example, Kaggle competitions have a reporting set that teams never access directly and that is used to rank their proposed solutions.

### Design B: Nested cross-validation

```text
Inner CV -> model selection
Outer CV -> performance estimation

```
This is useful when we want a resampling-based estimate of the complete selection procedure, especially with limited data. An additional external reporting set can be retained when the application requires it, but it is not necessary for understanding the separation of roles.

> **Key idea:** Cross-validation for tuning does not by itself imply nested cross-validation. Nesting is needed when cross-validation must serve both selection and performance-estimation roles.

## 10. Summary

<img src="cv6.png" width="800">

## 11. Exercises 📝

### 11.1 Deciding the correct way

Which of the following are correct ways of tuning and evaluating a machine learning pipeline? Select all that apply.

A. Split the data into training and reporting sets. Use cross-validation within the training set to select the hyperparameters, then fit the selected pipeline on the full training set and evaluate it once on the reporting set.

B. Split the data into training and reporting sets. Try several hyperparameter values and select the one that gives the best performance on the reporting set. Report this performance as the final result.

C. Use nested cross-validation, where the inner cross-validation is used for hyperparameter tuning and the outer cross-validation is used to estimate the performance of the complete modeling procedure.

D. Use cross-validation on the complete dataset to select the best hyperparameters. Then report the cross-validation performance obtained for those same selected hyperparameters as an unbiased estimate of the model's final performance.

YOUR ANSWERS HERE

### 11.2 Nested cross-validation in practice

Use nested cross-validation to tune and report the performance of our classification pipeline. Make sure that we tune the imputation strategy ("mean" vs. "median"), the number of features to select, and the value of $k$. Report more than one performance metric.

In [ ]:
# YOUR CODE HERE